# Projeto Final - 03. Silver para Gold (Dimensoes + Fato) + Carga no SQL
Ajustes desta versao:- **Schemas dedicados**: `staging` e `gold`, nada mais cai em `dbo`- **Dimensoes**: gravadas **direto** na tabela final (`gold.dim_*`), via overwrite
simples -- sao pequenas, baixa cardinalidade, nao precisam de staging nem MERGE- **Fato**: e a unica que passa por `staging.stg_fact_producao` e depois pela Stored
Procedure de upsert incremental
**Sobre o incremental (fato)**: o MERGE casa pela chave de negocio real (`sk_linha +
sk_maquina + sk_data`, que remonta pra `data_producao` **dentro** da linha). Um
arquivo processado atrasado -- por exemplo rodado no dia 08 mas com
`data_producao = 2026-07-07` -- ainda assim atualiza o registro do dia 07
corretamente, sem duplicar.

In [16]:
data_referencia = ""

StatementMeta(industry, 12, 17, Finished, Available, Finished, False)

In [17]:
caminho_silver = "abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/SILVER/ordens_producao_delta/"
df_silver = spark.read.format("delta").load(caminho_silver)
if data_referencia != "":
    df_silver = df_silver.filter(df_silver.data_producao == data_referencia)
df_silver.createOrReplaceTempView("ordens_silver")
print("Linhas na silver:", df_silver.count())

StatementMeta(industry, 12, 18, Finished, Available, Finished, False)

Linhas na silver: 168


## 1. Dimensoes e Fato (mesma logica de antes)

In [18]:
from pyspark.sql.functions import row_number, col, year, month, dayofmonth, date_format
from pyspark.sql.window import Window
dim_linha = (
    df_silver.select("linha_producao").distinct()
    .withColumn("sk_linha", row_number().over(Window.orderBy("linha_producao")))
    .select("sk_linha", "linha_producao")
)
dim_maquina = (
    df_silver.select("id_maquina", "linha_producao").distinct()
    .withColumn("sk_maquina", row_number().over(Window.orderBy("id_maquina")))
    .select("sk_maquina", "id_maquina", "linha_producao")
)
dim_data = (
    df_silver.select("data_producao").distinct()
    .withColumn("sk_data", row_number().over(Window.orderBy("data_producao")))
    .withColumn("ano", year("data_producao"))
    .withColumn("mes", month("data_producao"))
    .withColumn("dia", dayofmonth("data_producao"))
    .withColumn("nome_dia_semana", date_format("data_producao", "EEEE"))
    .select("sk_data", "data_producao", "ano", "mes", "dia", "nome_dia_semana")
)
df_gold = spark.sql("""
    SELECT
        linha_producao,
        id_maquina,
        data_producao,
        SUM(quantidade_produzida) AS total_produzido,
        SUM(quantidade_refugada) AS total_refugado,
        ROUND(SUM(quantidade_refugada) * 100.0 / SUM(quantidade_produzida), 2) AS taxa_refugo_pct,
        ROUND(AVG(produtividade_hora), 2) AS produtividade_media
    FROM ordens_silver
    GROUP BY linha_producao, id_maquina, data_producao
""")
fact_producao = (
    df_gold
    .join(dim_linha, "linha_producao")
    .join(dim_maquina.select("sk_maquina", "id_maquina"), "id_maquina")
    .join(dim_data.select("sk_data", "data_producao"), "data_producao")
    .select(
        "sk_linha", "sk_maquina", "sk_data",
        "total_produzido", "total_refugado", "taxa_refugo_pct", "produtividade_media"
    )
)
print("Dimensoes e fato construidas em memoria")

StatementMeta(industry, 12, 19, Finished, Available, Finished, False)

Dimensoes e fato construidas em memoria


## 2. Conexao JDBC + funcao auxiliar de DDL/DML

In [19]:
# Credenciais nunca em texto plano -- lidas do Azure Key Vault via linked service do Synapse.
# O Key Vault precisa estar configurado como linked service no workspace (kv-curso-azure).
usuario_sql = mssparkutils.credentials.getSecret("kv-curso-azure", "sql-server-user")
senha_sql = mssparkutils.credentials.getSecret("kv-curso-azure", "sql-server-password")
jdbc_url = (
    "jdbc:sqlserver://curso-azure-engenharia.database.windows.net:1433;"
    "database=free-sql-db-curso-azure;encrypt=true;trustServerCertificate=false"
)
propriedades_conexao = {
    "user": usuario_sql,
    "password": senha_sql,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
def executar_sql(comando, ignorar_se_ja_existe=False):
    """Executa DDL/DML direto no SQL via JDBC, usando o driver Java ja
    disponivel no cluster Spark (o mesmo usado pelo .write.jdbc)."""
    driver_manager = spark._sc._jvm.java.sql.DriverManager
    url_completa = f"{jdbc_url};user={propriedades_conexao['user']};password={propriedades_conexao['password']}"
    conexao = driver_manager.getConnection(url_completa)
    try:
        statement = conexao.createStatement()
        statement.execute(comando)
    except Exception as e:
        if ignorar_se_ja_existe and ("already exists" in str(e).lower() or "there is already an object" in str(e).lower()):
            print("Ja existe, ignorando.")
        else:
            raise
    finally:
        conexao.close()

StatementMeta(industry, 12, 20, Finished, Available, Finished, False)

## 3. Criacao dos schemas `staging` e `gold`

`CREATE SCHEMA` precisa ser o unico comando do batch em T-SQL -- por isso cada um
roda numa chamada separada.

In [20]:
executar_sql("CREATE SCHEMA staging", ignorar_se_ja_existe=True)
executar_sql("CREATE SCHEMA gold", ignorar_se_ja_existe=True)

print("Schemas staging e gold prontos")

StatementMeta(industry, 12, 21, Finished, Available, Finished, False)

Ja existe, ignorando.
Ja existe, ignorando.
Schemas staging e gold prontos


## 4. Dimensoes -- direto na camada final (gold), sem staging
Pequenas, baixa cardinalidade -- overwrite simples resolve, sem necessidade de
MERGE.

In [21]:
dim_linha.write.jdbc(jdbc_url, "gold.dim_linha", mode="overwrite", properties=propriedades_conexao)
dim_maquina.write.jdbc(jdbc_url, "gold.dim_maquina", mode="overwrite", properties=propriedades_conexao)
dim_data.write.jdbc(jdbc_url, "gold.dim_data", mode="overwrite", properties=propriedades_conexao)

print("Dimensoes gravadas direto em gold.dim_linha, gold.dim_maquina, gold.dim_data")

StatementMeta(industry, 12, 22, Finished, Available, Finished, False)

Dimensoes gravadas direto em gold.dim_linha, gold.dim_maquina, gold.dim_data


## 5. Fato -- vai pra staging (nao pra gold ainda)

In [27]:
(
    fact_producao.write
    .option("batchsize", 5000)
    .jdbc(jdbc_url, "staging.stg_fact_producao", mode="overwrite", properties=propriedades_conexao)
)    
print("Fato gravada em staging.stg_fact_producao")

StatementMeta(industry, 13, 2, Finished, Available, Finished, False)

NameError: name 'fact_producao' is not defined

## 6. DDL da tabela final gold.fact_producao (permanente, so criada se nao existir)

In [23]:
ddl_fact_producao = """
IF NOT EXISTS (
    SELECT 1 FROM sys.tables t
    JOIN sys.schemas s ON t.schema_id = s.schema_id
    WHERE t.name = 'fact_producao' AND s.name = 'gold'
)
CREATE TABLE gold.fact_producao (
    sk_linha INT,
    sk_maquina INT,
    sk_data INT,
    total_produzido INT,
    total_refugado INT,
    taxa_refugo_pct DECIMAL(5,2),
    produtividade_media DECIMAL(10,2),
    data_atualizacao DATETIME2 DEFAULT SYSUTCDATETIME(),
    PRIMARY KEY (sk_linha, sk_maquina, sk_data)
)
"""
executar_sql(ddl_fact_producao)
print("gold.fact_producao confirmada/criada")

StatementMeta(industry, 12, 24, Finished, Available, Finished, False)

gold.fact_producao confirmada/criada


## 7. Stored Procedure de upsert incremental -- so a fato
O MERGE casa por `sk_linha + sk_maquina + sk_data`. Como esses `sk_*` remontam
pra `data_producao` real (nao pra data de ingestao), um arquivo atrasado que carrega
dado de um dia anterior atualiza o registro certo, sem duplicar.

In [26]:
mssparkutils.notebook.exit("OK - dimensoes gravadas direto em gold, fato via staging + upsert")

StatementMeta(industry, 12, 27, Finished, Available, Finished, False)

ExitValue: OK - dimensoes gravadas direto em gold, fato via staging + upsert